# Split protocol

Required report-opening section: measure the performance gap between a random-utterance split and the chosen (conversation-level) split, using a baseline model. Grouping unit, ratios, label-distribution check, and duplicate-handling decision are justified in `src/splits.py`, `src/preprocessing.py`, and `notebooks/00_explore_data.ipynb`.

Baseline model used for this comparison (not the final classifier): logistic regression on TF-IDF of the flattened context. Chosen for speed — the comparison only needs a relative gap, not best absolute performance, so it's wasteful to run the full deep architecture multiple times just to pick a split.

Measured on the **3-class task** (Exploration / Comforting / Action), not 8-class: a coarser, less inherently noisy task gives a cleaner signal for detecting a leakage-driven gap with a simple baseline. Using the noisier 8-class task here would risk the comparison being dominated by task difficulty rather than the thing actually being measured — the effect of split choice.

In [ ]:
import sys
from pathlib import Path
import sklearn
sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
from scipy import stats
from data import load_and_clean, build_3class_examples, flatten_context
from data import apply_conversation_level_split, random_utterance_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

CONTRACTION_TOKEN_PATTERN = r"(?u)\b\w[\w']*\w\b"  # sklearn default r"(?u)\b\w\w+\b" splits on the apostrophe (we're -> "we","re"; that's -> "that", drops the "s") -- found while reading TF-IDF top-coefficient words; this keeps contractions whole, still drops single-char tokens like the default did.


In [2]:
convs = load_and_clean()
examples = build_3class_examples(convs)
print(f"Total 3-class examples: {len(examples)}")

Total 3-class examples: 14547


TF-IDF vocabulary is fit on train only, then applied (`.transform`, not `.fit_transform`) to test — per standard split-before-preprocessing practice, to avoid leaking test-set vocabulary into the vectorizer.

In [3]:
def evaluate_split(train, test, label="split"):
    """Returns (macro_f1, y_test, y_pred) -- y_test/y_pred kept for the single-seed bootstrap below."""
    X_train_text = [flatten_context(ex["context"], include_speaker_tags=False) for ex in train]
    X_test_text = [flatten_context(ex["context"], include_speaker_tags=False) for ex in test]
    y_train = [ex["gold_coarse"] for ex in train]
    y_test = [ex["gold_coarse"] for ex in test]

    vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), token_pattern=CONTRACTION_TOKEN_PATTERN)
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    print(f"--- {label} ---")
    print(f"Train: {len(train)}  Test: {len(test)}")
    print(f"Macro-F1 (test): {macro_f1:.4f}")
    print(classification_report(y_test, y_pred, zero_division=0))
    return macro_f1, np.array(y_test), np.array(y_pred)

In [4]:
conv_train, conv_val, conv_test = apply_conversation_level_split(convs, examples)
conv_macro_f1, conv_y_test, conv_y_pred = evaluate_split(conv_train, conv_test, label="Conversation-level split (seed=42)")

--- Conversation-level split (seed=42) ---
Train: 11672  Test: 1417
Macro-F1 (test): 0.4217
              precision    recall  f1-score   support

      Action       0.36      0.42      0.39       379
  Comforting       0.33      0.37      0.35       436
 Exploration       0.59      0.47      0.52       602

    accuracy                           0.43      1417
   macro avg       0.43      0.42      0.42      1417
weighted avg       0.45      0.43      0.43      1417



In [5]:
utt_train, utt_val, utt_test = random_utterance_split(examples)
utt_macro_f1, utt_y_test, utt_y_pred = evaluate_split(utt_train, utt_test, label="Random-utterance split (seed=42)")

--- Random-utterance split (seed=42) ---
Train: 11637  Test: 1456
Macro-F1 (test): 0.4341
              precision    recall  f1-score   support

      Action       0.40      0.43      0.41       431
  Comforting       0.33      0.35      0.34       430
 Exploration       0.58      0.51      0.54       595

    accuracy                           0.44      1456
   macro avg       0.44      0.43      0.43      1456
weighted avg       0.45      0.44      0.45      1456



In [6]:
gap = utt_macro_f1 - conv_macro_f1
print(f"Random-utterance macro-F1:   {utt_macro_f1:.4f}")
print(f"Conversation-level macro-F1: {conv_macro_f1:.4f}")
print(f"Gap (random - conversation), single seed=42: {gap:+.4f}")

Random-utterance macro-F1:   0.4341
Conversation-level macro-F1: 0.4217
Gap (random - conversation), single seed=42: +0.0124


## Significance test 1: single-seed bootstrap

The seed=42 test sets above contain different examples (not paired), so a paired test doesn't directly apply to them. First check: an independent two-sample bootstrap — each test set resampled with replacement many times, macro-F1 recomputed each time, giving an empirical sampling distribution per split and a 95% CI on the gap.

In [7]:
def bootstrap_macro_f1_gap(y_test_conv, y_pred_conv, y_test_utt, y_pred_utt, n_bootstrap=2000, seed=42):
    rng = np.random.RandomState(seed)
    n_conv = len(y_test_conv)
    n_utt = len(y_test_utt)

    f1_conv_samples = np.empty(n_bootstrap)
    f1_utt_samples = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx_conv = rng.randint(0, n_conv, n_conv)
        idx_utt = rng.randint(0, n_utt, n_utt)
        f1_conv_samples[i] = f1_score(y_test_conv[idx_conv], y_pred_conv[idx_conv], average="macro", zero_division=0)
        f1_utt_samples[i] = f1_score(y_test_utt[idx_utt], y_pred_utt[idx_utt], average="macro", zero_division=0)

    gap_samples = f1_utt_samples - f1_conv_samples
    ci_conv = np.percentile(f1_conv_samples, [2.5, 97.5])
    ci_utt = np.percentile(f1_utt_samples, [2.5, 97.5])
    ci_gap = np.percentile(gap_samples, [2.5, 97.5])
    p_value = np.mean(gap_samples <= 0)

    print(f"Conversation-level macro-F1: mean={f1_conv_samples.mean():.4f}  95% CI [{ci_conv[0]:.4f}, {ci_conv[1]:.4f}]")
    print(f"Random-utterance macro-F1:   mean={f1_utt_samples.mean():.4f}  95% CI [{ci_utt[0]:.4f}, {ci_utt[1]:.4f}]")
    print(f"Bootstrap gap (random - conversation): mean={gap_samples.mean():+.4f}  95% CI [{ci_gap[0]:+.4f}, {ci_gap[1]:+.4f}]")
    print(f"One-sided bootstrap p-value (H0: gap <= 0): {p_value:.4f}")
    return gap_samples, p_value

gap_samples, bootstrap_p_value = bootstrap_macro_f1_gap(conv_y_test, conv_y_pred, utt_y_test, utt_y_pred)

Conversation-level macro-F1: mean=0.4209  95% CI [0.3960, 0.4464]
Random-utterance macro-F1:   mean=0.4332  95% CI [0.4088, 0.4566]
Bootstrap gap (random - conversation): mean=+0.0122  95% CI [-0.0223, +0.0477]
One-sided bootstrap p-value (H0: gap <= 0): 0.2425


**Result**: gap = +0.0222, 95% CI [-0.0116, +0.0579], one-sided p=0.107 — CI includes 0, not significant.

## Significance test 2: repeated-seed paired test (sign test, Wilcoxon)

The bootstrap above only resamples *within* one fixed seed=42 split, so it can't separate "true effect of split choice" from "this particular random partition happened to land this way." To get a legitimate paired comparison for the sign test / Wilcoxon signed-rank test (both require matched pairs, unlike the unpaired bootstrap), the whole pipeline is repeated end-to-end for N different seeds. Each seed yields one conversation-level macro-F1 and one random-utterance macro-F1 — that pair (same seed, two split strategies, everything else held fixed) is the legitimate matched unit, not individual test examples.

In [8]:
def quick_macro_f1(train, test):
    X_train_text = [flatten_context(ex["context"], include_speaker_tags=False) for ex in train]
    X_test_text = [flatten_context(ex["context"], include_speaker_tags=False) for ex in test]
    y_train = [ex["gold_coarse"] for ex in train]
    y_test = [ex["gold_coarse"] for ex in test]
    vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), token_pattern=CONTRACTION_TOKEN_PATTERN)
    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return f1_score(y_test, y_pred, average="macro", zero_division=0)

N_REPEATS = 20
conv_f1s = []
utt_f1s = []
for i in range(N_REPEATS):
    ctrain, cval, ctest = apply_conversation_level_split(convs, examples, seed=i)
    utrain, uval, utest = random_utterance_split(examples, seed=i)
    conv_f1s.append(quick_macro_f1(ctrain, ctest))
    utt_f1s.append(quick_macro_f1(utrain, utest))

conv_f1s = np.array(conv_f1s)
utt_f1s = np.array(utt_f1s)
paired_diffs = utt_f1s - conv_f1s

print(f"Conversation-level macro-F1 across {N_REPEATS} seeds: mean={conv_f1s.mean():.4f}  std={conv_f1s.std():.4f}")
print(f"Random-utterance macro-F1 across {N_REPEATS} seeds:   mean={utt_f1s.mean():.4f}  std={utt_f1s.std():.4f}")
print(f"Mean paired gap (random - conversation): {paired_diffs.mean():+.4f}")

n_positive = int((paired_diffs > 0).sum())
n_negative = int((paired_diffs < 0).sum())
print(f"Paired differences: {n_positive} positive, {n_negative} negative, {N_REPEATS - n_positive - n_negative} tied")

sign_result = stats.binomtest(n_positive, n_positive + n_negative, p=0.5, alternative="greater")
print(f"Sign test, one-sided p (H0: gap <= 0): {sign_result.pvalue:.4f}")

wilcoxon_result = stats.wilcoxon(paired_diffs, alternative="greater")
print(f"Wilcoxon signed-rank test, one-sided p (H0: gap <= 0): {wilcoxon_result.pvalue:.4f}")

Conversation-level macro-F1 across 20 seeds: mean=0.4293  std=0.0136
Random-utterance macro-F1 across 20 seeds:   mean=0.4301  std=0.0117
Mean paired gap (random - conversation): +0.0008
Paired differences: 9 positive, 11 negative, 0 tied
Sign test, one-sided p (H0: gap <= 0): 0.7483
Wilcoxon signed-rank test, one-sided p (H0: gap <= 0): 0.4204


**Result**: mean paired gap = +0.0008 (conv=0.4293, utt=0.4301), 9 positive / 11 negative across 20 seeds, sign p=0.7483, Wilcoxon p=0.4204 — null.

## Split protocol conclusion

Single seed (42): gap = +0.0124 (random-utterance F1=0.4341 vs. conversation-level=0.4217), bootstrap 95% CI [-0.0223, +0.0477], p=0.2425 — not significant.

20 seeds: mean gap = +0.0008, 9/11 positive/negative split, sign p=0.7483, Wilcoxon p=0.4204 — null.

No significant leakage gap detected with this baseline. This does not overturn the conversation-level split: that choice rests on a model-independent argument (`src/splits.py`: 90.6% of conversations have same-speaker runs, supporter context windows overlap heavily across turns), not on this test finding a gap. Nor should the null be read as "too weak a model to detect leakage" — evidence on this is mixed. Soltaniani & Ghafari (arXiv:2601.22946) found Random Forest lose 26.97% MCC when duplicates are removed from a leaked secret-detection dataset, vs. only 7.22% for a transformer (GraphCodeBERT) — a low-capacity model *more*, not less, exploiting leakage. The more defensible reading here is that this dataset/task shows little exploitable literal overlap between splits, not that the baseline lacks capacity to find it.

*(Numbers above reflect a `token_pattern` fix to `TfidfVectorizer` -- sklearn's default tokenizer splits on apostrophes ("we're" -> "we"+"re", "that's" silently drops its "s"). Fixed to keep contractions whole; re-ran this notebook and 02_classifier_evaluation.ipynb. Old numbers: single-seed gap +0.0221 (utt=0.4360, conv=0.4138), 20-seed mean gap +0.0019, exact 10/10 split, sign p=0.588, Wilcoxon p=0.337 -- superseded, kept here for the audit trail. Conclusion (null result, no significant leakage) is unchanged.)*